In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [571]:
import pandas as pd
df_train = pd.read_csv('/content/drive/MyDrive/Crime_Detection/train_data.csv')
df_test = pd.read_csv('/content/drive/MyDrive/Crime_Detection/test_data.csv')

In [572]:
def check_mixed_types(df, column_name):
    """Checks if a column in a DataFrame has mixed data types."""

    unique_types = df[column_name].apply(type).unique()
    # print(unique_types)
    return len(unique_types)

def drop_rows_with_float(df, column_name):
    """Drops rows where the specified column has float type data."""
    df_filtered = df[df[column_name].apply(type) != float ]
    return df_filtered



#FILTER TRAIN SET

In [573]:
features = df_train.columns
for feature in features:
    mixed_types = check_mixed_types(df_train, feature)
    if mixed_types > 1:
        print(f"Column '{feature}' has '{mixed_types}' data types.")

df_filtered = drop_rows_with_float(df_train, 'maritalstatus')
df_filtered = drop_rows_with_float(df_filtered, 'race')
df_filtered = drop_rows_with_float(df_filtered, 'sex')

df_train_filtered = df_filtered
features = df_filtered.columns
for feature in features:
    mixed_types = check_mixed_types(df_train_filtered, feature)
    if mixed_types > 1:
        print(f"Column '{feature}' has '{mixed_types}' data types.")
df_train_filtered.shape

Column 'maritalstatus' has '2' data types.
Column 'race' has '2' data types.
Column 'sex' has '2' data types.


(20505, 14)

#FILTER TEST SET

In [574]:
df_filtered = drop_rows_with_float(df_test, 'maritalstatus')
df_filtered = drop_rows_with_float(df_filtered, 'race')
df_filtered = drop_rows_with_float(df_filtered, 'sex')

df_test_filtered = df_filtered
features = df_filtered.columns
for feature in features:
    mixed_types = check_mixed_types(df_test_filtered, feature)
    if mixed_types > 1:
        print(f"Column '{feature}' has '{mixed_types}' data types.")



In [575]:
df_train['race'].value_counts()

,count
race,
White,17693
Black,1925
Asian-Pac-Islander,644
Amer-Indian-Eskimo,213
Other,161


In [576]:
categorical_features = ['workclass', 'education', 'maritalstatus', 'occupation', 'relationship', 'race', 'sex', 'native']
continuous_features = ['age', 'educationno', 'capitalgain', 'capitalloss', 'hoursperweek']
ohe_features = ['maritalstatus','race','sex']
fe_features = ['workclass','education','occupation','relationship','native']

In [577]:
df_train['occupation'].value_counts()

,count
occupation,
Craft-repair,2851
Prof-specialty,2834
Exec-managerial,2753
Adm-clerical,2615
Sales,2529
Other-service,2234
Machine-op-inspct,1341
Transport-moving,1104
Handlers-cleaners,949


# ONE HOT ENCODING

##OHE for Train set


In [578]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# --- One-Hot Encoding ---
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe_encoded = pd.DataFrame(ohe.fit_transform(df_train_filtered[ohe_features]))
ohe_encoded.columns = ohe.get_feature_names_out(ohe_features)
# --- Frequency Encoding ---
fe_encoded = df_train_filtered[fe_features].apply(lambda x: x.map(x.value_counts(normalize=True)))
# --- Combine and Drop ---
df_train_filtered = df_train_filtered.drop(ohe_features + fe_features, axis=1)
df_train_filtered = pd.concat([df_train_filtered, ohe_encoded, fe_encoded], axis=1)

##OHE for Test set

In [579]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# --- One-Hot Encoding ---
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe_encoded = pd.DataFrame(ohe.fit_transform(df_test_filtered[ohe_features]))
ohe_encoded.columns = ohe.get_feature_names_out(ohe_features)

# --- Frequency Encoding ---
fe_encoded = df_test_filtered[fe_features].apply(lambda x: x.map(x.value_counts(normalize=True)))

# --- Combine and Drop ---
df_test_filtered = df_test_filtered.drop(ohe_features + fe_features, axis=1)
df_test_filtered = pd.concat([df_test_filtered, ohe_encoded, fe_encoded], axis=1)

#Split into X and y

In [580]:
X_train=df_train_filtered.drop(['Possibility','educationno'],axis=1)
y_train=df_train_filtered['Possibility']
X_test=df_test_filtered.drop(['Possibility','educationno'],axis=1)
y_test=df_test_filtered['Possibility']

In [581]:
X_train.shape,y_train.shape

((21091, 23), (21091,))

In [582]:
df2=pd.concat([X_train,y_train],axis=1)

In [583]:
df2=df2.drop_duplicates()

In [584]:
df2_test=pd.concat([X_test,y_test],axis=1)

In [585]:
df2_test=df2_test.drop_duplicates()

In [586]:
df2.shape,df2_test.shape

((19838, 24), (8657, 24))

In [587]:
df2['profit']=df2['capitalgain']-df2['capitalloss']
df2_test['profit']=df2_test['capitalgain']-df2_test['capitalloss']
df2=df2.drop(['capitalgain','capitalloss'],axis=1)
df2_test=df2_test.drop(['capitalgain','capitalloss'],axis=1)

<ipython-input-587-4978e33d25dd>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2_test['profit']=df2_test['capitalgain']-df2_test['capitalloss']


In [588]:
df2.isna().sum()

,0
age,32
hoursperweek,344
maritalstatus_Divorced,581
maritalstatus_Married-AF-spouse,581
maritalstatus_Married-civ-spouse,581
maritalstatus_Married-spouse-absent,581
maritalstatus_Never-married,581
maritalstatus_Separated,581
maritalstatus_Widowed,581
race_Amer-Indian-Eskimo,581


In [589]:
df2_test.isna().sum()

,0
age,29
hoursperweek,172
maritalstatus_Divorced,259
maritalstatus_Married-AF-spouse,259
maritalstatus_Married-civ-spouse,259
maritalstatus_Married-spouse-absent,259
maritalstatus_Never-married,259
maritalstatus_Separated,259
maritalstatus_Widowed,259
race_Amer-Indian-Eskimo,259


In [590]:
df2 = df2.dropna(subset=['Possibility'])
df2_test = df2_test.dropna(subset=['Possibility'])

In [591]:
X_train=df2.drop('Possibility',axis=1)
y_train=df2['Possibility']
X_test=df2_test.drop('Possibility',axis=1)
y_test=df2_test['Possibility']

In [592]:
X_train.shape,y_train.shape,X_test.shape,y_test.shape

((19806, 22), (19806,), (8628, 22), (8628,))

##Displaying first five rows with Nan Values

In [593]:
null_rows = X_train[X_train.isnull().any(axis=1)]
if not null_rows.empty:
  print(null_rows.head(5))
else:
  print("No rows with null values found.")

      age  hoursperweek  maritalstatus_Divorced  \
65   23.0           NaN                     0.0   
75   40.0           NaN                     0.0   
192  32.0           NaN                     0.0   
315  20.0           NaN                     0.0   
347  61.0           NaN                     0.0   

     maritalstatus_Married-AF-spouse  maritalstatus_Married-civ-spouse  \
65                               0.0                               1.0   
75                               0.0                               0.0   
192                              0.0                               1.0   
315                              0.0                               1.0   
347                              0.0                               0.0   

     maritalstatus_Married-spouse-absent  maritalstatus_Never-married  \
65                                   0.0                          0.0   
75                                   0.0                          1.0   
192                          

##Filled Nan values with respective median values

In [594]:
for column in X_train.columns:
  if column == 'hoursperweek':
    X_train[column].fillna(0, inplace=True) # Fill NaNs in 'hoursperweek' with 0
  else:
    X_train[column].fillna(X_train[column].median(), inplace=True) # Fill NaNs in other columns with median

In [595]:
null_counts = y_train.isna().sum()
print(null_counts)

0


In [596]:
for column in X_test.columns:
  if column == 'hoursperweek':
    X_test[column].fillna(0, inplace=True) # Fill NaNs in 'hoursperweek' with 0
  else:
    X_test[column].fillna(X_test[column].median(), inplace=True)


In [597]:
null_counts_test = X_test.isna().sum()
print(null_counts_test)

age                                    0
hoursperweek                           0
maritalstatus_Divorced                 0
maritalstatus_Married-AF-spouse        0
maritalstatus_Married-civ-spouse       0
maritalstatus_Married-spouse-absent    0
maritalstatus_Never-married            0
maritalstatus_Separated                0
maritalstatus_Widowed                  0
race_Amer-Indian-Eskimo                0
race_Asian-Pac-Islander                0
race_Black                             0
race_Other                             0
race_White                             0
sex_Female                             0
sex_Male                               0
workclass                              0
education                              0
occupation                             0
relationship                           0
native                                 0
profit                                 0
dtype: int64


##Scaling the processed train and test data with StandardScaler

In [598]:
common_features = X_train.columns.intersection(X_test.columns)
X_train = X_train[common_features]
X_test = X_test[common_features]
X_test = X_test[X_train.columns]
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

In [599]:
X_test_scaled

array([[-0.80147175,  1.11959032, -0.39975691, ..., -0.0750399 ,
         0.3342074 , -0.14006516],
       [ 0.18635684, -0.03241306, -0.39975691, ...,  1.12832411,
        -3.14408803, -0.14006516],
       [ 1.09819861,  0.73558919, -0.39975691, ...,  1.12832411,
        -3.13106721, -0.14006516],
       ...,
       [ 0.18635684,  1.50359145, -0.39975691, ..., -0.0750399 ,
         0.3342074 , 12.76141844],
       [-1.105419  , -0.80041531, -0.39975691, ..., -0.0750399 ,
         0.3342074 , -0.14006516],
       [-0.95344537, -0.03241306, -0.39975691, ..., -0.0750399 ,
         0.3342074 , -0.14006516]])

In [600]:
X_test_scaled.shape,y_test.shape,X_train_scaled.shape,y_train.shape

((8628, 22), (8628,), (19806, 22), (19806,))

## Finding Feature importances using Random Forest for further processing

In [601]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train_scaled, y_train)
importances = model.feature_importances_
feature_importance_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

print(feature_importance_df)

                                Feature  Importance
0                                   age    0.214750
21                               profit    0.155827
19                         relationship    0.153180
18                           occupation    0.119011
1                          hoursperweek    0.105878
17                            education    0.094089
16                            workclass    0.046498
20                               native    0.018396
4      maritalstatus_Married-civ-spouse    0.014121
6           maritalstatus_Never-married    0.013473
2                maritalstatus_Divorced    0.010461
14                           sex_Female    0.009725
15                             sex_Male    0.009654
13                           race_White    0.008510
11                           race_Black    0.006483
8                 maritalstatus_Widowed    0.004387
7               maritalstatus_Separated    0.004340
10              race_Asian-Pac-Islander    0.004298
5   maritals

In [602]:
important_features=['age','profit','relationship','occupation','hoursperweek','education','workclass']
X_train_imp=X_train_scaled[:,[X_train.columns.get_loc(c) for c in important_features]] # Get the integer index of each column name and use that to slice the NumPy array.
X_test_imp=X_test_scaled[:,[X_test.columns.get_loc(c) for c in important_features]] # Get the integer index of each column name and use that to slice the NumPy array.

## KNN

In [603]:
from sklearn.neighbors import KNeighborsClassifier
model = KNeighborsClassifier(n_neighbors=20)
model.fit(X_train_imp, y_train)

KNeighborsClassifier(n_neighbors=20)

In [604]:
from sklearn.metrics import accuracy_score
y_pred=model.predict(X_test_imp)
accuracy=accuracy_score(y_test,y_pred)
print(f"Accuracy:{accuracy}")

Accuracy:0.8196569309225776


## Random Forest Classifier

In [605]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Initialize the Random Forest model with some good hyperparameters
rf_model = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    bootstrap=True,
    random_state=42
)

# Perform 5-fold cross-validation
scores = cross_val_score(rf_model, X_train_imp, y_train, cv=5, scoring='accuracy')

# Print the cross-validation scores and the average accuracy
print("Cross-validation scores:", scores)
print(f"Average accuracy: {scores.mean()}")

Cross-validation scores: [0.84174659 0.84145418 0.83842464 0.83413279 0.83665741]
Average accuracy: 0.8384831231207963


##XGB with Optimal Paramters

In [606]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

# Initialize the XGBoost model with some good hyperparameters
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    random_state=42
)

scores = cross_val_score(xgb_model, X_train_imp, y_train, cv=5, scoring='accuracy')

print("Cross-validation scores:", scores)
print(f"Average accuracy: {scores.mean()}")

Cross-validation scores: [0.87657749 0.86771017 0.86291341 0.86139864 0.86619541]
Average accuracy: 0.8669590215861591


##Randomized Search with RandomForest

In [607]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# Define the parameter grid
param_dist = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

# Create a Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)

# Set up RandomizedSearchCV
random_search = RandomizedSearchCV(
    rf_model,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring='accuracy',
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# Fit the model
random_search.fit(X_train_imp, y_train)

# Get the best parameters and the best score
print("Best parameters found: ", random_search.best_params_)
print("Best accuracy found: ", random_search.best_score_)

# Get the best model
best_rf_model = random_search.best_estimator_

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters found:  {'n_estimators': 150, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 10, 'bootstrap': True}
Best accuracy found:  0.8553970113197314


##Voting of RandomForest and KNN

In [608]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier

rf_model = RandomForestClassifier(n_estimators=100, criterion="gini", max_depth=15)
knn_model = KNeighborsClassifier(n_neighbors=20)

ensemble_model = VotingClassifier(estimators=[
    ('rf', rf_model), ('knn', knn_model)
], voting='soft')

# Fit the ensemble model
ensemble_model.fit(X_train_imp, y_train)

# Predict and calculate accuracy
y_pred = ensemble_model.predict(X_test_imp)
accuracy = accuracy_score(y_test, y_pred)
print(f"Ensemble Model Accuracy (RF and KNN): {accuracy}")

Ensemble Model Accuracy (RF and KNN): 0.8358831710709318


## Voting of RF, XGB and KNN

In [609]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# Initialize the individual models
rf_model = RandomForestClassifier(n_estimators=100, criterion="gini", max_depth=15)
knn_model = KNeighborsClassifier(n_neighbors=20)
xgb_model = XGBClassifier(
     n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1
)

ensemble_model = VotingClassifier(estimators=[
    ('rf', rf_model), ('knn', knn_model), ('xgb', xgb_model)
], voting='soft')

# Fit the ensemble model
ensemble_model.fit(X_train_imp, y_train)

# Predict and calculate accuracy
y_pred = ensemble_model.predict(X_test_imp)
accuracy = accuracy_score(y_test, y_pred)
print(f"Ensemble Model Accuracy (RF, KNN, XGBoost): {accuracy}")

Ensemble Model Accuracy (RF, KNN, XGBoost): 0.8441121928604544


## AdaBoost from Scratch

In [610]:
import numpy as np

class AdaBoostClassifier:
    def __init__(self, base_estimator, n_estimators=200,learning_rate=0.01):
        self.base_estimator = base_estimator
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.estimators_ = []
        self.estimator_weights_ = np.zeros(self.n_estimators, dtype=np.float64)
        self.estimator_errors_ = np.ones(self.n_estimators, dtype=np.float64)

    def fit(self, X, y):
        n_samples = X.shape[0]
        sample_weights = np.full(n_samples, (1 / n_samples))

        for i in range(self.n_estimators):
            estimator = self.base_estimator
            estimator.fit(X, y, sample_weight=sample_weights)

            y_pred = estimator.predict(X)
            incorrect = (y_pred != y)
            error = np.dot(incorrect, sample_weights) / np.sum(sample_weights)
            decay=0.99
            alpha = self.learning_rate * (decay ** i) * 0.5 * np.log((1.0 - error) / error)
            sample_weights *= np.exp(alpha * incorrect * ((sample_weights > 0) | (alpha < 0)))

            self.estimators_.append(estimator)
            self.estimator_weights_[i] = alpha
            self.estimator_errors_[i] = error

        return self

    def predict(self, X):
        n_samples = X.shape[0]
        y_pred = np.zeros((n_samples, 1))

        for i, estimator in enumerate(self.estimators_):
            y_pred_i = estimator.predict(X).reshape((n_samples, 1))
            y_pred += self.estimator_weights_[i] * y_pred_i

        y_pred = np.sign(y_pred).flatten()
        return y_pred


from sklearn.tree import DecisionTreeClassifier
X_train, y_train = X_train_imp, y_train
X_test, y_test = X_test_imp, y_test
base_estimator = DecisionTreeClassifier(max_depth=3)
adaboost_model = AdaBoostClassifier(base_estimator=base_estimator, n_estimators=50)
adaboost_model.fit(X_train, y_train)

y_pred = adaboost_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"AdaBoost Accuracy: {accuracy}")

AdaBoost Accuracy: 0.808066759388039


##Gradient Boosting From Scratch

In [628]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

class GradientBoostingClassifier:
    def __init__(self, n_estimators=200, learning_rate=0.05, max_depth=10):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y):
        # Initialize with log(odds)
        self.initial_prediction = np.log(np.sum(y == 1) / np.sum(y == 0))
        predictions = np.full(len(X), self.initial_prediction)

        for _ in range(self.n_estimators):
            # Calculate negative gradient (residuals)
            residuals = y - 1 / (1 + np.exp(-predictions))

            # Fit a tree to the residuals
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)

            # Update predictions
            predictions += self.learning_rate * tree.predict(X)

    def predict(self, X):
        predictions = np.full(len(X), self.initial_prediction)

        for tree in self.trees:
            predictions += self.learning_rate * tree.predict(X)

        # Convert log(odds) to probabilities and then to class labels
        probabilities = 1 / (1 + np.exp(-predictions))
        return (probabilities > 0.5).astype(int)

X_train, y_train = X_train_imp, y_train
X_test, y_test = X_test_imp, y_test

gb_model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=10)
gb_model.fit(X_train, y_train)

y_pred = gb_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Gradient Boosting Accuracy: {accuracy}")

Gradient Boosting Accuracy: 0.8350718590635141


In [630]:
import pandas as pd
!pip install pandas

# Concatenate X_train_imp and y_train
X_train_final = pd.concat([pd.DataFrame(X_train_imp), pd.DataFrame(y_train)], axis=1)

# Save to CSV
X_train_final.to_csv('X_train_final.csv', index=False)
X_test_final = pd.concat([pd.DataFrame(X_test_imp), pd.DataFrame(y_test)], axis=1)

# Save to CSV
X_test_final.to_csv('X_test_final.csv', index=False)